# Claude Managed Agents로 SRE 장애 대응 에이전트 만들기

## 들어가며

새벽 3시에 프로덕션 알림이 울리면 누군가는 로그를 뒤지고, 알맞은 런북을 찾고, 잘못된 설정을 추적하고, PR을 열고, 승인을 받아야 합니다. 에이전트가 그 첫 단계를 대신해 두면, 여러분이 키보드 앞에 앉을 즈음에는 검토만 하면 되는 수정안이 준비되어 있습니다. 알맞은 맥락이 주어지고 최종 판단은 사람이 내린다는 전제하에서요.

[Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview)는 그것을 손쉽게 만들 수 있는 확장 가능한 인프라, 샌드박싱, 보안 요소를 제공합니다. 이 튜토리얼에서는 이들을 하나로 엮습니다.

- 모의 **PagerDuty 웹훅**이 API 호출 한 번으로 Claude Managed Agent를 촉발합니다.
- **스킬(Skill)**이 팀의 런북 관례를 에이전트에 가르쳐, 어디를 봐야 할지 알게 합니다.
- 내장 `bash`/`read`/`edit` 도구로 샌드박스 안에서 로그와 인프라 코드를 조사합니다.
- **커스텀 도구**로 풀 리퀘스트를 열고 병합 전에 사람의 승인을 요청합니다. 그 호출은 여러분의 코드가 처리하므로 "PR을 연다"는 것이 실제로 무엇을 뜻하는지는 여러분이 정합니다.
- **Anthropic 콘솔**이 모든 단계를 자동으로 기록해 완전한 관측 가능성을 제공합니다.

아래 내용은 `ANTHROPIC_API_KEY`만 있으면 모두 실행됩니다. PagerDuty, GitHub, Datadog은 로컬 픽스처로 대체해 두어 Managed Agents 부분에 집중할 수 있게 했습니다. 마지막 절에서 각 목을 실제 서비스로 바꾸는 방법을 보여 줍니다.

### 배울 내용

- 스킬을 업로드해 Claude Managed Agent에 붙이기
- 내장 툴셋과 여러분의 애플리케이션이 처리하는 커스텀 도구를 함께 쓰기
- 웹훅 페이로드로 세션 시작하기
- 파괴적 동작을 사람 승인 뒤에 두기
- 콘솔에서 세션 전체 추적 읽기

### 사전 준비

환경에 `ANTHROPIC_API_KEY`를 설정한 뒤 의존성을 설치하세요:

In [1]:
%pip install -q "anthropic>=0.91.0" python-dotenv

In [2]:
import json
import os
import time
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv
from utilities import wait_for_idle_status

load_dotenv()
client = Anthropic()
MODEL = os.getenv("COOKBOOK_MODEL", "claude-opus-4-6")
FIXTURE = Path("example_data/sre")

## 1. 런북 스킬 업로드하기

[**스킬**](https://platform.claude.com/docs/en/managed-agents/skills)은 플랫폼이 점진적 공개 방식으로 에이전트의 컨텍스트에 마운트하는 작은 파일 시스템 묶음입니다. 에이전트는 처음에 한 줄짜리 설명만 보고, 관련이 있을 때만 본문을 읽습니다. 시스템 프롬프트에 두기에는 부적절한 팀 관례를 담기 좋은 자리입니다.

아래 샘플 스킬은 실제 팀 플레이북이 그러하듯 규칙 하나를 담고 있습니다. *인프라를 건드리기 전에 런북을 확인하라*는 것입니다. Skills API로 한 번 업로드해 두고, 필요한 에이전트마다 ID로 참조합니다.

In [3]:
# A real skill is usually a folder on disk (SKILL.md plus any helper
# scripts or reference docs) that you zip and upload. For this tutorial
# the SKILL.md is small enough to keep inline.
RUNBOOK_SKILL = """\
---
name: incident-runbooks
description: How to triage production incidents using the team runbooks.
---

# Incident runbooks

When an alert references a service, locate that service's recent logs
and identify the failure signature (the repeating error class, exit
code, or status pattern).

Consult the team runbooks before proposing any fix. Runbooks are
organised by failure signature — for example `oom.md`, `5xx.md`,
`latency.md`. Each one lists the triage steps for that class of
failure and the configuration that usually needs to change.

Any fix to infrastructure code must be opened as a pull request that
cites the runbook you followed. Do not patch live resources directly.
"""

skill = client.beta.skills.create(
    display_title="incident-runbooks",
    files=[("incident-runbooks/SKILL.md", RUNBOOK_SKILL.encode(), "text/markdown")],
)
print(f"skill: {skill.id} (version {skill.latest_version})")

skill: skill_01WPWHALbtEVBUWG6mHa7Tna (version 1775588716519983)


## 2. 에이전트 만들기

에이전트의 `tools` 목록은 세 종류의 능력을 결합합니다.

- [`agent_toolset_20260401`](https://platform.claude.com/docs/en/managed-agents/tools) — 샌드박스 *안에서* 실행되는 내장 `bash`, `read`, `grep`, `edit` 등의 도구. 에이전트는 이것으로 조사합니다.
- 1단계에서 만든 런북 **스킬**.
- 세 개의 **커스텀 도구** — `open_pull_request`, `request_approval`, `merge_pull_request` — 에이전트가 호출하되 *여러분의 애플리케이션*이 실행합니다. 에이전트가 샌드박스 밖의 시스템에 닿는 방법이자, 사람을 루프에 넣는 방법입니다.

시스템 프롬프트에는 페르소나와 워크플로만 담겨 있습니다. 알림 자체는 첫 사용자 이벤트로 도착하므로, 같은 에이전트가 어떤 장애든 처리합니다.

In [4]:
SRE_SYSTEM_PROMPT = """\
You are an on-call SRE agent. Each user message is a PagerDuty alert
payload. Triage it to root cause and ship the minimal safe fix.

The session workspace contains the recent logs, the infrastructure
repo, and the team runbooks for the alerting service. Explore it to
find what you need.

Workflow for every alert:
1. Read the logs and identify the failure signature.
2. Find the root cause in the infrastructure repo, save a copy of the
   original file, edit it in place, then produce a unified diff with
   `diff -u`.
3. open_pull_request(title, body, diff) with the fix.
4. request_approval(summary) and wait for the human's decision.
5. Only if the result is "approved", merge_pull_request(pr_number).
   Otherwise stop and report.

Never call merge_pull_request unless request_approval returned
"approved". Keep the fix minimal — do not refactor unrelated config.
"""

agent = client.beta.agents.create(
    name="cookbook-sre-responder",
    model=MODEL,
    system=SRE_SYSTEM_PROMPT,
    skills=[{"type": "custom", "skill_id": skill.id, "version": skill.latest_version}],
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        },
        {
            "type": "custom",
            "name": "open_pull_request",
            "description": "Open a pull request against the infra repo with the proposed fix.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "body": {"type": "string"},
                    "diff": {"type": "string", "description": "Unified diff of the change."},
                },
                "required": ["title", "body", "diff"],
            },
        },
        {
            "type": "custom",
            "name": "request_approval",
            "description": "Ask the on-call human to approve the proposed PR before merging.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                },
                "required": ["summary"],
            },
        },
        {
            "type": "custom",
            "name": "merge_pull_request",
            "description": "Merge an approved pull request.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "pr_number": {"type": "integer"},
                },
                "required": ["pr_number"],
            },
        },
    ],
)
print(f"agent: {agent.id} v{agent.version}")

agent: agent_011CZpw3Y76Vu4t2j2QEosVa v1


## 3. 환경 만들고 데이터 마운트하기

에이전트가 조사하려면 작업 공간에 세 가지가 필요합니다. 최근 서비스 로그, 인프라 저장소, 팀 런북입니다. 각각을 Files API로 업로드하고 `resources`로 나열해, 시스템 프롬프트가 기대하는 경로에 모든 세션마다 마운트되게 합니다. 에이전트가 자기 파일 시스템만 있으면 되므로 `limited` 네트워킹 클라우드 환경으로 충분합니다.

이 노트북을 `ANTHROPIC_API_KEY`만으로 실행할 수 있게 하려고, 인프라 "저장소"는 `memory: 128Mi`라는 지나치게 낮은 한도가 담긴 매니페스트 한 개로 대신했습니다. 프로덕션에서는 이 업로드를 실제 저장소를 샌드박스로 곧바로 클론하는 `github_repository` 리소스로 바꾸게 됩니다:

```python
{
    "type": "github_repository",
    "url": "https://github.com/your-org/infra",
    "authorization_token": os.environ["GITHUB_TOKEN"],
    "checkout": {"type": "branch", "name": "main"},
    "mount_path": "infra",
}
```

In [5]:
env = client.beta.environments.create(
    name="cookbook-sre-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)


def upload(path: Path, mime: str) -> str:
    with path.open("rb") as f:
        return client.beta.files.upload(file=(path.name, f, mime)).id


log_id = upload(FIXTURE / "logs/checkout-svc.log", "text/plain")
manifest_id = upload(FIXTURE / "infra/k8s/checkout-deploy.yaml", "text/yaml")
runbook_id = upload(FIXTURE / "runbooks/oom.md", "text/markdown")

RESOURCES = [
    {"type": "file", "file_id": log_id, "mount_path": "logs/checkout-svc.log"},
    {"type": "file", "file_id": manifest_id, "mount_path": "infra/k8s/checkout-deploy.yaml"},
    {"type": "file", "file_id": runbook_id, "mount_path": "runbooks/oom.md"},
]
print(f"environment: {env.id}")

environment: env_01R6hmJkd6BhpPotXnoC7rqU


## 4. 장애 알림 처리하기

아래 핸들러가 여러분이 실제로 배포할 단 하나의 함수입니다. 장애가 발생했을 때 알림 시스템이 호출하는 Flask나 FastAPI 라우트입니다. 에이전트와 환경을 참조하는 세션을 만들고, 데이터를 마운트하고, 알림 JSON을 첫 `user.message` 이벤트로 보냅니다. 이 예제는 [PagerDuty V3 웹훅](https://developer.pagerduty.com/docs/webhooks-overview) 페이로드를 사용하지만, JSON을 POST할 수 있는 어떤 호출기든 똑같이 동작합니다. 여기서는 픽스처로 핸들러를 직접 호출합니다.

In [6]:
def handle_pagerduty_webhook(payload: dict) -> str:
    incident = payload["event"]["data"]
    session = client.beta.sessions.create(
        environment_id=env.id,
        agent={"type": "agent", "id": agent.id, "version": agent.version},
        resources=RESOURCES,
        title=f"[{incident['service']['summary']}] {incident['title']}",
    )
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [{"type": "text", "text": json.dumps(payload, indent=2)}],
            }
        ],
    )
    return session.id


with (FIXTURE / "alert.json").open() as f:
    alert = json.load(f)

session_id = handle_pagerduty_webhook(alert)
print(f"session: {session_id}")

session: sesn_011CZpw3gtC691y7qmaLNmLM


## 5. 에이전트의 커스텀 도구 호출에 응답하기

여기서 내장 도구와 여러분의 커스텀 도구가 만납니다. 에이전트의 `read`/`bash`/`edit` 호출은 컨테이너에서 실행되어 이벤트 로그에 `agent.tool_use`로 나타납니다. 그것이 조사 과정이고, 여러분은 출력하기만 하면 됩니다. 하지만 에이전트가 여러분의 커스텀 도구를 호출하면 세션이 `stop_reason.type == "requires_action"`과 함께 `idle` 상태가 되어 *여러분의 애플리케이션*이 `user.custom_tool_result`로 응답하기를 기다립니다.

아래 루프는 `events.list`를 폴링하며, `open_pull_request`와 `merge_pull_request`는 로컬 리스트에 기록하는 것으로 즉석에서 응답하고(이것이 GitHub 목입니다), `request_approval`이 도착하면 **반환**합니다. 그 하나는 사람이 필요하기 때문입니다.

프로덕션에서 "사람이 필요하다"는 것은 보통 *Slack에 올린다*는 뜻입니다. 에이전트의 요약을 온콜 채널에 **승인** 버튼과 함께 올리고, 누군가 클릭하면 결과를 돌려보냅니다. [`slack_data_bot` 쿡북](slack_data_bot.ipynb)이 그 Bolt 연결을 보여 줍니다. 여기서는 노트북이 자체 완결되도록 다음 셀에서 직접 승인하겠습니다.

In [7]:
prs: list[dict] = []
pending_approvals: list[dict] = []
seen_events: set[str] = set()


def handle_custom_tool(name: str, args: dict) -> dict:
    if name == "open_pull_request":
        n = len(prs) + 1
        prs.append({"number": n, "merged": False, **args})
        print(f"\n── PR #{n}: {args['title']} ──")
        return {"pr_number": n, "url": f"mock://infra/pull/{n}"}
    if name == "merge_pull_request":
        prs[args["pr_number"] - 1]["merged"] = True
        return {"merged": True}
    raise ValueError(f"unhandled tool {name}")


def run_until_approval_or_end(session_id: str) -> str | None:
    """Poll the session's event log, servicing custom tools, until either
    a request_approval call arrives (return its event_id so the caller
    can respond) or the agent ends its turn (return None)."""
    custom_calls: dict[str, object] = {}
    responded: set[str] = set()
    while True:
        idle_stop = None
        for ev in client.beta.sessions.events.list(session_id):
            if ev.id in seen_events:
                continue
            seen_events.add(ev.id)
            if ev.type == "agent.message":
                for block in ev.content:
                    if block.type == "text":
                        print(block.text, end="")
            elif ev.type == "agent.tool_use":
                print(f"\n  [{ev.name}]")
            elif ev.type == "agent.custom_tool_use":
                custom_calls[ev.id] = ev
                print(f"\n→ {ev.name}")
            elif ev.type == "session.status_idle":
                idle_stop = ev.stop_reason
            elif ev.type == "session.status_terminated":
                return None
        if idle_stop is None:
            time.sleep(1.0)
            continue
        if idle_stop.type == "end_turn":
            return None
        if idle_stop.type == "requires_action":
            for event_id in idle_stop.event_ids:
                if event_id in responded:
                    continue
                call = custom_calls[event_id]
                if call.name == "request_approval":
                    pending_approvals.append({"event_id": event_id, **call.input})
                    return event_id
                result = handle_custom_tool(call.name, call.input)
                client.beta.sessions.events.send(
                    session_id,
                    events=[
                        {
                            "type": "user.custom_tool_result",
                            "custom_tool_use_id": event_id,
                            "content": [{"type": "text", "text": json.dumps(result)}],
                        }
                    ],
                )
                responded.add(event_id)
        time.sleep(1.0)


approval_event_id = run_until_approval_or_end(session_id)

I'll help you triage this production incident. The alert shows checkout-svc pods are crash-looping with 7 restarts in 5 minutes. Let me start by exploring the workspace to find logs and the infrastructure configuration.
  [bash]
Let me check the session directory for logs and infrastructure files:
  [bash]
Perfect! Let me examine the logs first to identify the failure signature:
  [read]
Perfect! I've identified the failure signature. The logs show a clear **OutOfMemoryError** pattern:

**Failure Signature:**
- Service starts and warms pricing cache (14,092 entries)
- Heap memory grows rapidly: 101MB → 118MB → 121MB (against 128MB limit)
- `pricing.recompute` function attempts allocation during garbage collection pause (412ms GC pause observed)
- Container gets OOMKilled (exit 137) after ~2 minutes
- Service restarts and immediately repeats the cycle

Now let me examine the deployment configuration to identify the root cause:
  [read]
**Root Cause Analysis:**

The deployment has memory

에이전트가 로그를 읽고, 스킬을 통해 `OOMKilled` 시그니처를 `runbooks/oom.md`에 대응시키고, `infra/k8s/checkout-deploy.yaml`에서 128Mi 메모리 한도를 찾아 수정하고, PR을 연 뒤 이제 여러분을 기다리고 있습니다. 다음은 여러분의 `#oncall` Slack 채널에 올라올 메시지입니다:

In [8]:
pr = prs[0]
print(pr["body"], "\n")
print(pr["diff"])
print("\n" + "─" * 60)
print("APPROVAL REQUESTED:", pending_approvals[0]["summary"])

## Issue
checkout-svc pods are in a CrashLoopBackOff state due to OutOfMemoryError. The service consistently crashes after ~2 minutes with 7 restarts in the last 5 minutes.

## Root Cause
The deployment had memory limits set to 128Mi, which is insufficient for the pricing cache operation:
- Pricing cache warms with 14,092 entries during startup
- Heap pressure builds to 118-121MB (92-94% of limit) within 90 seconds
- pricing.recompute fails to allocate 8MB, causing OOMKilled (exit 137)
- Service restarts and repeats the cycle

## Fix
Increase memory allocation to provide adequate headroom:
- **Memory request:** 128Mi → 256Mi
- **Memory limit:** 128Mi → 512Mi

This provides 4x headroom for the pricing cache and normal operations while remaining resource-efficient (512Mi limit is standard for Java/similar workloads with caching).

## Verification
The fix addresses the immediate OOMKilled pattern in logs and aligns memory resources with the actual cache size and operational requirements. 

## 6. 승인하고 에이전트가 병합하게 하기

`request_approval`의 결과로 `"approved"`를 돌려보냅니다. 에이전트가 재개해 `merge_pull_request`를 호출하고 턴을 끝냅니다. Slack 버전에서는 이 전송이 버튼 클릭 핸들러 안에서 일어나며, 페이로드는 동일합니다.

In [9]:
client.beta.sessions.events.send(
    session_id,
    events=[
        {
            "type": "user.custom_tool_result",
            "custom_tool_use_id": approval_event_id,
            "content": [{"type": "text", "text": json.dumps({"decision": "approved"})}],
        }
    ],
)

run_until_approval_or_end(session_id)
print(f"\n\nPR #{pr['number']} merged: {prs[0]['merged']}")

Great! The fix has been approved. Now merging:
→ merge_pull_request
## ✅ Incident Resolved

**Summary:**
- **Status:** MERGED (PR #1)
- **Failure:** checkout-svc crash-loop with OOMKilled (exit 137)
- **Root Cause:** Memory limit of 128Mi was insufficient for pricing cache (14,092 entries)
- **Fix Applied:** 
  - Memory request: 128Mi → 256Mi
  - Memory limit: 128Mi → 512Mi

**What Happened:**
1. Service loaded 14k+ pricing cache entries during startup
2. Heap grew to 118-121MB within ~2 minutes (92-94% of 128Mi limit)
3. pricing.recompute failed to allocate 8MB, triggering OutOfMemoryError
4. Container was OOMKilled and restarted, repeating the cycle

**Expected Outcome:**
With 512Mi limit and 256Mi request, the service will have sufficient memory headroom for:
- Pricing cache operations
- Normal request processing
- JVM garbage collection pauses
- No more CrashLoopBackOff

The deployment update will trigger a rolling restart of the 3 replicas, allowing pods to spawn with the new memo

## 7. 콘솔에서 실행 검토하기

이 조사는 Managed Agents 세션으로 실행되었기 때문에, 위의 모든 단계(파일 읽기, `bash` diff, 매니페스트 수정, 세 번의 커스텀 도구 호출, 여러분이 보낸 승인)가 세션의 이벤트로 보존됩니다. [콘솔](https://platform.claude.com/)의 **Managed Agents → Sessions**에서 열면 별도 계측 없이 전체 감사 기록을 볼 수 있습니다:

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/sre/console_session.png" alt="장애 대응 실행의 콘솔 세션 화면" width="700" />

### 정리

세션과 만들어 둔 리소스를 아카이브합니다.

In [10]:
wait_for_idle_status(client, session_id)
client.beta.sessions.archive(session_id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
client.beta.skills.versions.delete(skill.latest_version, skill_id=skill.id)
client.beta.skills.delete(skill.id)
print("archived")

archived


## 다음 단계: 프로덕션 연결

세 가지만 바꾸면 노트북에서 실제 온콜로 넘어갑니다.

**Slack에서 승인하기.** `request_approval`이 도착하면 Block Kit 버튼과 함께 온콜 채널에 올리고, 액션 핸들러에서 `user.custom_tool_result`를 돌려보냅니다. [`slack_data_bot` 쿡북](slack_data_bot.ipynb)이 Bolt 앱 설정을 다루며, 승인에 해당하는 부분은 짧습니다:

```python
def post_for_approval(session_id, event_id, summary):
    slack.client.chat_postMessage(
        channel=ONCALL_CHANNEL,
        text=summary,
        blocks=[
            {"type": "section", "text": {"type": "mrkdwn", "text": summary}},
            {"type": "actions", "elements": [
                {"type": "button", "text": {"type": "plain_text", "text": "Approve"},
                 "action_id": "approve", "value": f"{session_id}:{event_id}"},
                {"type": "button", "text": {"type": "plain_text", "text": "Reject"},
                 "action_id": "reject", "value": f"{session_id}:{event_id}"},
            ]},
        ],
    )

@slack.action("approve")
def on_approve(ack, body):
    ack()
    session_id, event_id = body["actions"][0]["value"].split(":")
    client.beta.sessions.events.send(
        session_id,
        events=[{"type": "user.custom_tool_result",
                 "custom_tool_use_id": event_id,
                 "content": [{"type": "text",
                              "text": json.dumps({"decision": "approved"})}]}],
    )
```

폴링 루프를 아예 없애려면 콘솔에서 `session.requires_action`에 대한 웹훅을 등록하세요. 에이전트가 멈추는 순간 플랫폼이 여러분의 엔드포인트를 호출하고, 거기서 Slack에 올리면 됩니다.

**목 대신 GitHub 쓰기.** `open_pull_request` / `merge_pull_request` 커스텀 도구를 없애고 대신 GitHub MCP 서버를 에이전트에 주되, 토큰은 볼트에 저장해 코드에 절대 드러나지 않게 합니다. [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb)가 사용자별 자격 증명을 다룹니다.

```python
agent = client.beta.agents.create(
    ...,
    mcp_servers=[{"type": "url", "name": "github",
                  "url": "https://api.githubcopilot.com/mcp/"}],
    tools=[{"type": "agent_toolset_20260401", ...},
           {"type": "mcp_toolset", "server_name": "github"},
           {"type": "custom", "name": "request_approval", ...}],
)
session = client.beta.sessions.create(..., vault_ids=[github_vault.id])
```

**픽스처 대신 실시간 로그 쓰기.** 환경 설정으로 `DD_API_KEY` / `DD_APP_KEY`를 전달하고, 에이전트가 마운트된 파일을 읽는 대신 `bash`에서 Datadog Logs API를 `curl`하게 하세요.

> 호스팅되는 Managed Agents 런타임 대신 로컬 Agent SDK로 같은 문제를 푼 사례는 [Agent SDK 사이트 신뢰성 에이전트](https://github.com/anthropics/claude-cookbooks/blob/main/claude_agent_sdk/03_The_site_reliability_agent.ipynb)도 참고하세요.

## 배운 것

- 외부 이벤트로 세션을 촉발하기 — PagerDuty 웹훅에서 온 API 호출 한 번이 전체 실행을 시작했습니다.
- **스킬**을 붙여 에이전트에 팀의 관례를 알려 주기.
- **리소스**로 데이터 마운트하기: 코드에는 `github_repository`, 로그와 런북에는 `file`.
- **커스텀 도구**로 여러분의 앱을 다시 호출하고, `requires_action`으로 사람 승인 뒤에 동작을 두기.
- **콘솔** 세션 화면에서 전체 감사 기록을 별도 비용 없이 얻기.

목을 GitHub MCP, Slack 승인 버튼, 실시간 로그로 바꾸면 온콜에 투입할 준비가 끝납니다.